# Load Data

In [11]:
import pandas as pd
import os

# 1) Load processed CSVs
demo = pd.read_csv("../data/processed/demo_clean.csv")
bpx = pd.read_csv("../data/processed/bpx_clean.csv")
bmx = pd.read_csv("../data/processed/bmx_clean.csv")

print("=============== DEMO HEAD ===============\n")
print(demo.head())
print("\n=============== BPX HEAD ===============\n")
print(bpx.head())
print("\n=============== BMX HEAD ===============\n")
print(bmx.head())

=============== DEMO HEAD ===============

       SEQN  SDDSRVYR  RIDSTATR  RIAGENDR  RIDAGEYR  RIDAGEMN  RIDRETH1  \
0  130378.0      12.0       2.0       1.0      43.0       NaN       5.0   
1  130379.0      12.0       2.0       1.0      66.0       NaN       3.0   
2  130380.0      12.0       2.0       2.0      44.0       NaN       2.0   
3  130381.0      12.0       2.0       2.0       5.0       NaN       5.0   
4  130382.0      12.0       2.0       1.0       2.0       NaN       3.0   

   RIDRETH3  RIDEXMON  RIDEXAGM  ...  DMDHRGND  DMDHRAGZ  DMDHREDZ  DMDHRMAZ  \
0       6.0       2.0       NaN  ...       NaN       NaN       NaN       NaN   
1       3.0       2.0       NaN  ...       NaN       NaN       NaN       NaN   
2       2.0       1.0       NaN  ...       NaN       NaN       NaN       NaN   
3       7.0       1.0      71.0  ...       2.0       2.0       2.0       3.0   
4       3.0       2.0      34.0  ...       2.0       2.0       3.0       1.0   

   DMDHSEDZ      WTINT2YR

# Cleaning and Scaling Steps:
- Merged demo + BPX + BMI on `SEQN`
- Dropped missing values
- Scaled age and BP using StandardScaler
- Saved to `cleaned_normalized.csv`

In [12]:
# Clean & Normalize NHANES Data


# 2) Clean: Handle Missing Values:
# Merge all data:
dataframes = [demo, bpx, bmx] # list of datasets to merge
cleaned = dataframes[0]
for df in dataframes[1:]:
    cleaned = cleaned.merge(df, on="SEQN", how="inner")

# Drop missing values
cleaned = cleaned.dropna(subset=["BPXOSY1", "BPXODI1", "RIDAGEYR", "RIAGENDR", "RIDRETH1", "BMXBMI"])

# keep only relevant columns
cleaned = cleaned[["SEQN", "BPXOSY1", "BPXODI1", "RIDAGEYR", "RIAGENDR", "RIDRETH1", "BMXBMI"]]


# 3) Normalize: Scale Age, BMI & BP
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
cleaned[["RIDAGEYR_scaled","BPXOSY1_scaled", "BPXODI1_scaled", "BMXBMI_scaled"]] = scaler.fit_transform(
    cleaned[["RIDAGEYR", "BPXOSY1", "BPXODI1", "BPXODI1"]])


# 4) Encode Ethnicity
cleaned = pd.get_dummies(cleaned, columns=["RIDRETH1"], prefix="eth", drop_first=True)
print("=============== Cleaned Data HEAD ===============\n")
print(cleaned.head())

# 5) Save Cleaned Data:
cleaned.to_csv("../data/processed/cleaned_normalized.csv", index=False)

=============== Cleaned Data HEAD ===============

       SEQN  BPXOSY1  BPXODI1  RIDAGEYR  RIAGENDR  BMXBMI  RIDAGEYR_scaled  \
0  130378.0    135.0     98.0      43.0       1.0    27.0        -0.083921   
1  130379.0    121.0     84.0      66.0       1.0    33.5         0.931883   
2  130380.0    111.0     79.0      44.0       2.0    29.7        -0.039756   
3  130386.0    110.0     72.0      34.0       1.0    30.2        -0.481410   
4  130387.0    143.0     76.0      68.0       2.0    42.6         1.020214   

   BPXOSY1_scaled  BPXODI1_scaled  BMXBMI_scaled  eth_2.0  eth_3.0  eth_4.0  \
0        0.852785        2.128180       2.128180        0        0        0   
1        0.095689        0.948017       0.948017        0        1        0   
2       -0.445093        0.526530       0.526530        1        0        0   
3       -0.499172       -0.063551      -0.063551        0        0        0   
4        1.285411        0.273638       0.273638        0        1        0   

   et